In [1]:
import os, random
from collections import defaultdict

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision import datasets
from PIL import Image
from tqdm import tqdm

# ------------------------------
# 1. Config
# ------------------------------
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 1
TARGET_PER_CLASS = 2000
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------
# 2. Dataset Class
# ------------------------------
class LungDataset(Dataset):
    def __init__(self, root_dir, transform=None, selected_indices=None):
        self.base = datasets.ImageFolder(root=root_dir)
        self.samples = self.base.samples
        self.classes = self.base.classes
        self.class_to_idx = self.base.class_to_idx
        self.transform = transform
        if selected_indices:
            self.samples = [self.samples[i] for i in selected_indices]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# ------------------------------
# 3. Transforms
# ------------------------------
train_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.3, hue=0.02),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ------------------------------
# 4. Paths
# ------------------------------
DATA_ROOT = "DataSet"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR  = os.path.join(DATA_ROOT, "valid")
TEST_DIR = os.path.join(DATA_ROOT, "test")

# ------------------------------
# 5. Base Datasets
# ------------------------------
train_base = LungDataset(TRAIN_DIR)
val_base   = LungDataset(VAL_DIR)
test_base  = LungDataset(TEST_DIR)

NUM_CLASSES = len(train_base.classes)
print(f"✅ Found {NUM_CLASSES} classes: {train_base.classes}")


# ------------------------------
# 6. Data Augmentation + Save New Images
# ------------------------------
from collections import defaultdict
from torchvision import transforms as T
from PIL import Image
import shutil

class_to_idxs = defaultdict(list)
for i, (path, lbl) in enumerate(train_base.samples):
    class_to_idxs[lbl].append((i, path))

AUGMENT_DIR = os.path.join(DATA_ROOT, "augmented")
os.makedirs(AUGMENT_DIR, exist_ok=True)

image_id = 0
aug_train_idx = []

augmentations = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.3),
])

# Save augmented images only for classes with < TARGET_PER_CLASS
for lbl, idx_paths in class_to_idxs.items():
    cls_name = train_base.classes[lbl]
    save_dir = os.path.join(AUGMENT_DIR, cls_name)
    os.makedirs(save_dir, exist_ok=True)

    original_count = len(idx_paths)
    required = TARGET_PER_CLASS - original_count
    sampled = random.choices(idx_paths, k=required) if required > 0 else []

    for _, path in sampled:
        img = Image.open(path).convert("RGB")
        img_aug = augmentations(img)
        save_path = os.path.join(save_dir, f"aug_{image_id}.jpg")
        img_aug.save(save_path)
        image_id += 1

    for i, _ in idx_paths:
        aug_train_idx.append(i)  # original indices retained

# Custom dataset that includes original + augmented
class AugmentedLungDataset(Dataset):
    def __init__(self, original_dataset, aug_dir, transform=None, selected_indices=None):
        self.samples = original_dataset.samples.copy()
        self.transform = transform
        self.class_to_idx = original_dataset.class_to_idx

        for root, _, files in os.walk(aug_dir):
            cls_name = os.path.basename(root)
            if cls_name in self.class_to_idx:
                lbl = self.class_to_idx[cls_name]
                for file in files:
                    if file.endswith(".jpg"):
                        self.samples.append((os.path.join(root, file), lbl))

        if selected_indices:
            self.samples = [self.samples[i] for i in selected_indices]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# Final dataset and loader
train_ds = AugmentedLungDataset(train_base, AUGMENT_DIR, train_transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f"📦 Final Train Dataset (with Augmentation): {len(train_ds)} samples")


val_ds   = LungDataset(VAL_DIR,   val_transform)
test_ds  = LungDataset(TEST_DIR,  val_transform)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"📦 Dataset sizes - Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

# =============================================================
# 8. Custom Swin Transformer (full shifted-window version)
# =============================================================
import torch
from torch import nn, einsum
import numpy as np
from einops import rearrange

# -- helper modules --------------------------------------------------

class CyclicShift(nn.Module):
    def __init__(self, displacement):
        super().__init__()
        self.displacement = displacement

    def forward(self, x):
        return torch.roll(x, shifts=(self.displacement, self.displacement), dims=(1, 2))


class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) + x


class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)


class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim),
        )

    def forward(self, x):
        return self.net(x)


def create_mask(window_size, displacement, upper_lower, left_right):
    mask = torch.zeros(window_size ** 2, window_size ** 2)
    if upper_lower:
        mask[-displacement * window_size:, :-displacement * window_size] = float('-inf')
        mask[:-displacement * window_size, -displacement * window_size:] = float('-inf')
    if left_right:
        mask = rearrange(mask, '(h1 w1) (h2 w2) -> h1 w1 h2 w2', h1=window_size, h2=window_size)
        mask[:, -displacement:, :, :-displacement] = float('-inf')
        mask[:, :-displacement, :, -displacement:] = float('-inf')
        mask = rearrange(mask, 'h1 w1 h2 w2 -> (h1 w1) (h2 w2)')
    return mask


def get_relative_distances(window_size):
    indices = torch.tensor(np.array([[x, y] for x in range(window_size) for y in range(window_size)]))
    distances = indices[None, :, :] - indices[:, None, :]
    return distances


# -- windowed attention --------------------------------------------------

class WindowAttention(nn.Module):
    def __init__(self, dim, heads, head_dim, shifted, window_size, relative_pos_embedding):
        super().__init__()
        inner_dim = head_dim * heads
        self.heads = heads
        self.scale = head_dim ** -0.5
        self.window_size = window_size
        self.relative_pos_embedding = relative_pos_embedding
        self.shifted = shifted

        if shifted:
            disp = window_size // 2
            self.cyclic_shift = CyclicShift(-disp)
            self.cyclic_back_shift = CyclicShift(disp)
            self.upper_lower_mask = nn.Parameter(
                create_mask(window_size, disp, upper_lower=True, left_right=False),
                requires_grad=False)
            self.left_right_mask = nn.Parameter(
                create_mask(window_size, disp, upper_lower=False, left_right=True),
                requires_grad=False)

        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        if relative_pos_embedding:
            self.relative_indices = get_relative_distances(window_size) + window_size - 1
            self.pos_embedding = nn.Parameter(torch.randn(2 * window_size - 1, 2 * window_size - 1))
        else:
            self.pos_embedding = nn.Parameter(torch.randn(window_size ** 2, window_size ** 2))
        self.to_out = nn.Linear(inner_dim, dim)

    def forward(self, x):
        if self.shifted:
            x = self.cyclic_shift(x)

        b, H, W, _, h = *x.shape, self.heads
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        nw_h, nw_w = H // self.window_size, W // self.window_size

        q, k, v = map(
            lambda t: rearrange(t, 'b (nh wh) (nw ww) (h d) -> b h (nh nw) (wh ww) d',
                                h=h, wh=self.window_size, ww=self.window_size),
            qkv)

        dots = einsum('b h w i d, b h w j d -> b h w i j', q, k) * self.scale

        if self.relative_pos_embedding:
            dots += self.pos_embedding[
                self.relative_indices[:, :, 0],
                self.relative_indices[:, :, 1]
            ]
        else:
            dots += self.pos_embedding

        if self.shifted:
            dots[:, :, -nw_w:] += self.upper_lower_mask
            dots[:, :, nw_w - 1::nw_w] += self.left_right_mask

        attn = dots.softmax(dim=-1)
        out = einsum('b h w i j, b h w j d -> b h w i d', attn, v)
        out = rearrange(
            out,
            'b h (nh nw) (wh ww) d -> b (nh wh) (nw ww) (h d)',
            nh=nw_h, nw=nw_w, wh=self.window_size, ww=self.window_size
        )
        out = self.to_out(out)

        if self.shifted:
            out = self.cyclic_back_shift(out)
        return out


class SwinBlock(nn.Module):
    def __init__(self, dim, heads, head_dim, mlp_dim, shifted, window_size, relative_pos_embedding):
        super().__init__()
        self.attn = Residual(PreNorm(dim, WindowAttention(
            dim, heads, head_dim, shifted, window_size, relative_pos_embedding)))
        self.mlp = Residual(PreNorm(dim, FeedForward(dim, mlp_dim)))

    def forward(self, x):
        x = self.attn(x)
        x = self.mlp(x)
        return x

class PatchMerging(nn.Module):
    def __init__(self, in_channels, out_channels, downscaling_factor):
        super().__init__()
        self.downscaling_factor = downscaling_factor
        self.unfold = nn.Unfold(
            kernel_size=downscaling_factor,
            stride=downscaling_factor,
            padding=0
        )
        self.linear = nn.Linear(in_channels * downscaling_factor**2,
                                out_channels)

    def forward(self, x):
        b, c, h, w = x.shape
        df = self.downscaling_factor
        nh, nw = h // df, w // df

        # unfold → (b, c*df*df, nh*nw)
        x = self.unfold(x)  
        # reshape to (b, nh, nw, c*df*df)
        x = x.view(b, c * df * df, nh, nw).permute(0, 2, 3, 1)
        return self.linear(x)


class StageModule(nn.Module):
    def __init__(self, in_ch, hid_dim, layers, down_factor, heads, head_dim, window_size, relative_pos_embedding):
        super().__init__()
        assert layers % 2 == 0
        self.merge = PatchMerging(in_ch, hid_dim, down_factor)
        self.blocks = nn.ModuleList()
        for _ in range(layers // 2):
            self.blocks.append(nn.ModuleList([
                SwinBlock(hid_dim, heads, head_dim, hid_dim * 4, False, window_size, relative_pos_embedding),
                SwinBlock(hid_dim, heads, head_dim, hid_dim * 4, True,  window_size, relative_pos_embedding),
            ]))

    def forward(self, x):
        x = self.merge(x)
        # x shape: (b, new_h, new_w, hid_dim)
        for regular, shifted in self.blocks:
            x = regular(x)
            x = shifted(x)
        return x.permute(0, 3, 1, 2)


class SwinTransformer(nn.Module):
    def __init__(self, *,
                 hidden_dim, layers, heads,
                 channels=3, num_classes=NUM_CLASSES,
                 head_dim=32, window_size=7,
                 downscaling_factors=(4, 2, 2, 2),
                 relative_pos_embedding=True):
        super().__init__()
        self.stage1 = StageModule(channels,      hidden_dim,     layers[0],
                                  downscaling_factors[0], heads[0], head_dim,
                                  window_size, relative_pos_embedding)
        self.stage2 = StageModule(hidden_dim,     hidden_dim*2,   layers[1],
                                  downscaling_factors[1], heads[1], head_dim,
                                  window_size, relative_pos_embedding)
        self.stage3 = StageModule(hidden_dim*2,   hidden_dim*4,   layers[2],
                                  downscaling_factors[2], heads[2], head_dim,
                                  window_size, relative_pos_embedding)
        self.stage4 = StageModule(hidden_dim*4,   hidden_dim*8,   layers[3],
                                  downscaling_factors[3], heads[3], head_dim,
                                  window_size, relative_pos_embedding)

        self.norm = nn.LayerNorm(hidden_dim * 8)
        self.head = nn.Linear(hidden_dim * 8, num_classes)

    def forward(self, img):
        x = self.stage1(img)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = x.mean(dim=[2, 3])       # global avg pooling
        return self.head(self.norm(x))


# ---------- instantiate and move to device ----------

model = SwinTransformer(
    hidden_dim=96,
    layers=(2,2,6,2),
    heads=(3,6,12,24),
    channels=3,
    num_classes=NUM_CLASSES,
    head_dim=32,
    window_size=7
).to(DEVICE)

print(f"✅ Swin Transformer ready → parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f} M")


✅ Found 3 classes: ['Bengin cases', 'Malignant cases', 'normal']
📦 Final Train Dataset (with Augmentation): 6664 samples
📦 Dataset sizes - Train: 6664, Val: 83, Test: 214
✅ Swin Transformer ready → parameters: 27.5 M


In [ ]:
from collections import Counter
label_counts = Counter([label for _, label in train_ds])
print(label_counts)  # Should print {0: 1000, 1: 1000} if binary classes


### Training 

In [ ]:
import os
import torch
from tqdm import tqdm

# ------------------------------
# 0. Checkpoint Config
# ------------------------------
CHECKPOINT_PATH = "swin_checkpoint.pth"
RESUME          = True           # set False to start fresh even if a ckpt exists

# ------------------------------
# 1. Loss, Optimizer, Scheduler
# ------------------------------
criterion  = nn.CrossEntropyLoss()
optimizer  = optim.AdamW(model.parameters(), lr=LR)
scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# ------------------------------
# 2. Optionally Resume
# ------------------------------
start_epoch   = 0
best_val_acc  = 0.0

if RESUME and os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optim_state"])
    scheduler.load_state_dict(ckpt["sched_state"])
    best_val_acc = ckpt["best_val_acc"]
    start_epoch  = ckpt["epoch"] + 1
    print(f"✅ Resumed from epoch {ckpt['epoch']} | best_val_acc={best_val_acc:.4f}")
else:
    print("ℹ️  No checkpoint found or RESUME=False — starting fresh.")

# ------------------------------
# 3. Train + Validate + Save
# ------------------------------
for epoch in range(start_epoch, NUM_EPOCHS):
    print(f"\n🔄 Epoch {epoch+1}/{NUM_EPOCHS}")
    model.train()
    running_loss = correct_preds = total_preds = 0

    for images, labels in tqdm(train_loader, desc="  • Training"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct_preds += (outputs.argmax(1) == labels).sum().item()
        total_preds   += labels.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    train_acc  = correct_preds / total_preds
    print(f"    🟢 Train  | loss={train_loss:.4f}  acc={train_acc:.4f}")

    # ---------- Validation ----------
    model.eval()
    val_loss = val_correct = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="  • Validate", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_loss /= len(val_loader.dataset)
    val_acc   = val_correct / len(val_loader.dataset)
    print(f"    🔵 Val    | loss={val_loss:.4f}  acc={val_acc:.4f}")

    # ---------- Checkpoint ----------
    improved = val_acc > best_val_acc
    if improved:
        best_val_acc = val_acc
        torch.save(
            {
                "epoch":        epoch,
                "model_state":  model.state_dict(),
                "optim_state":  optimizer.state_dict(),
                "sched_state":  scheduler.state_dict(),
                "best_val_acc": best_val_acc,
            },
            CHECKPOINT_PATH,
        )
        print(f"    💾 Saved new best checkpoint (acc={best_val_acc:.4f})")

    scheduler.step()
    print(f"    🔄 LR stepped -> {scheduler.get_last_lr()[0]:.6f}")

print("\n🎉 Training complete.")


### INFERENCE ONLY LOADING

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE)["model_state"])
model.eval()

### Confusion Matrix & Testing

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

# Run on test data
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_base.classes)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix on Test Set")
plt.show()

# Optional: Classification Report
print("\n🧾 Classification Report:\n", classification_report(all_labels, all_preds, target_names=train_base.classes))


### LIME

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries
import numpy as np
from matplotlib import pyplot as plt

# 1. Load one image without transform
raw_test_ds = LungDataset(TEST_DIR, transform=None)
sample_img, sample_label = raw_test_ds[0]
img_np = np.array(sample_img)

# 2. Define predict_proba
def predict_proba(images_np):
    model.eval()
    with torch.no_grad():
        images_resized = torch.nn.functional.interpolate(
            torch.tensor(images_np).permute(0, 3, 1, 2).float(),
            size=(224, 224), mode='bilinear', align_corners=False
        ) / 255.0
        images_resized = T.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225])(images_resized)
        images_resized = images_resized.to(DEVICE)
        outputs = model(images_resized)
        return outputs.softmax(1).cpu().numpy()


# 3. Explain with LIME
explainer = lime_image.LimeImageExplainer()
explanation = explainer.explain_instance(
    image=img_np,
    classifier_fn=predict_proba,
    top_labels=1,
    hide_color=0,
    num_samples=1000
)

# 4. Visualize
temp, mask = explanation.get_image_and_mask(
    label=explanation.top_labels[0],
    positive_only=False,
    hide_rest=False,
    num_features=10,
    min_weight=0.0
)

plt.figure(figsize=(6, 6))
plt.imshow(mark_boundaries(temp / 255.0, mask))
plt.title(f"LIME Explanation for class: {train_base.classes[sample_label]}")
plt.axis('off')
plt.show()
